# <b> Lunes </b>

<b> probabilidad | ¿El estimador “ingenuo” realmente gana? Aquí quiero que veas el fenómeno mediante simulación. </b>

Supón que el verdadero alpha es:

$$ \mu=0.0003. $$

Realiza 5_000 experimentos. En cada experimento genera solamente 50 observaciones:

> r = np.random.normal(   loc=0.0003,scale=0.01,size=50)

Calcula:

> mu_hat = r.mean()

y un estimador shrinkage muy simple:

> mu_shrunk = 0.5 * mu_hat

Compara sobre los 5,000 experimentos:

$$ Bias,\qquad Variance,\qquad MSE $$

de ambos estimadores respecto al verdadero $\mu$.

Construye sólo:

| Estimator   | Bias | Variance | MSE |
| ----------- | ---: | -------: | --: |
| Sample mean |      |          |     |
| Shrunk mean |      |          |     |

Después repite cambiando únicamente:

size=1000

No optimices $ \lambda $ 

La pregunta importante es: ¿por qué shrinkage puede ser especialmente valioso con muestras pequeñas y perder parte de su ventaja conforme aumenta \(N\)?

Relaciona explícitamente tu explicación con:

$SE(μ^)∝ \frac{1}{ \sqrt{N}} N$
	​

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)

true_mu = 0.0003
sigma = 0.01
n_experimentos = 5_000

def comparar_estimadores(n_observaciones):
    # Cada fila representa un experimento distinto
    retornos = rng.normal(
        loc=true_mu,
        scale=sigma,
        size=(n_experimentos, n_observaciones)
    )

    # Un mu_hat por experimento
    mu_hat = retornos.mean(axis=1)

    # Estimador shrinkage fijado por el enunciado
    mu_shrunk = 0.5 * mu_hat

    def metricas(estimador):
        return {
            "Bias": estimador.mean() - true_mu,
            "Variance": estimador.var(ddof=1),
            "MSE": ((estimador - true_mu) ** 2).mean()
        }

    return pd.DataFrame(
        {
            "Sample mean": metricas(mu_hat),
            "Shrunk mean": metricas(mu_shrunk)
        }
    ).T
resultados_50 = comparar_estimadores(n_observaciones=50)
resultados_50

,Bias,Variance,MSE
Sample mean,0.000001,2.003386e-06,2.002987e-06
Shrunk mean,-0.000149,5.008464e-07,5.230453e-07


In [ ]:
resultados_1000 = comparar_estimadores(n_observaciones=1000)
resultados_1000

,Bias,Variance,MSE
Sample mean,0.000008,1.000599e-07,1.001067e-07
Shrunk mean,-0.000146,2.501497e-08,4.630065e-08


: 

# Martes

<b> Research | Un resultado espectacular con poca información. Compara dos estrategias ficticias: </b>

$$ A:\quad\hat\mu=12\text{ bps},\ SE=8 $$ $$ B:\quad\hat\mu=7\text{ bps},\ SE=2. $$

Para ambas usa el mismo prior:

$$ \mu\sim N(0,5^2)\text{ bps}. $$

Escribe una función pequeña:

>`def posterior_normal(mu_prior, sd_prior, mu_hat, se):`
>
>       `...`

que devuelva:

- posterior_mean
- posterior_sd
- P(mu > 0)

Usa:  $$\sigma_{\text{post}}^2 = \left( \frac{1}{\sigma_0^2} + \frac{1}{SE^2} \right)^{-1}.$$

|                |  A |  B |
| -------------- | -: | -: |
| Raw estimate   | 12 |  7 |
| SE             |  8 |  2 |
| Posterior mean |    |    |
| Posterior SD   |    |    |
| \(P(\mu>0)\)   |    |    |


Ahora decide cuál de estas afirmaciones puedes defender:

$$ \text{“A tiene mayor estimated alpha”} $$

versus

$$ \text{“A tiene evidencia más fuerte de alpha”.} $$

No son equivalentes.  

- <b> R:  </b> A tiene el estimate puntual más alto, pero B tiene evidencia más confiable de alpha positivo.

Finalmente responde: ¿por qué ordenar estrategias únicamente por $ \hat\mu $  o  Sharpe observado favorece resultados extremos y potencialmente ruidosos? 

- <b> R: </b> Ordenar sólo por $\hat\mu $  o por Sharpe observado favorece resultados extremos porque ambos son estimaciones muestrales. Una estrategia con mucho ruido puede obtener un retorno o Sharpe excepcional por azar, especialmente con pocas observaciones; al elegir la cifra más alta, seleccionas también parte de ese ruido.

In [16]:
import numpy as np 
import pandas as pd 
from scipy.stats import norm
strate_a = np.array([12,8])
strate_b = np.array([7,2])
# prior data 
mu_prior = 0 
sigma_prior = 5

def posterior_normal(mu_prior, sd_prior, mu_hat, se):
    posterior_variance = 1 / (1 / sd_prior**2 +1 / se**2)

    posterior_mean = posterior_variance * (mu_prior / sd_prior**2 +mu_hat / se**2)

    posterior_sd = np.sqrt(posterior_variance)

    p_mu_positive = norm.cdf(posterior_mean / posterior_sd)

    return {
        "Raw estimate": mu_hat,
        "SE": se,
        "Posterior mean": posterior_mean,
        "Posterior SD": posterior_sd,
        "P(mu > 0)": p_mu_positive
    }

a =  posterior_normal(mu_prior, sigma_prior,strate_a[0], strate_a[1])
b = posterior_normal(mu_prior, sigma_prior,strate_b[0], strate_b[1])
df_resultados = pd.DataFrame([a, b], index = ['strategy_A', 'strategy_B'])
df_resultados

,Raw estimate,SE,Posterior mean,Posterior SD,P(mu > 0)
strategy_A,12,8,3.370787,4.239992,0.786693
strategy_B,7,2,6.034483,1.856953,0.999422


<b> Portfolio/Risk + Quant Desk | Una view también tiene incertidumbre. Regresa a tu arquitectura de escenarios: </b>

$$ (R,p)\rightarrow\text{Views}\rightarrow(R,q). $$

Tu Research Engine produce:

> “Equities deberían superar bonds durante el próximo mes.”

Una interfaz ingenua guarda:

>`view:`
>
>`E[R_equity - R_bond] >= 0`

Pero compara dos fuentes posibles.

Research A: 15 años de evidencia, múltiples regímenes, efecto estable.

Research B: 35 observaciones recientes, efecto grande pero muy inestable.

Ambas producen exactamente la misma restricción matemática.

Diseña ahora un contrato de view con sólo:

- statement
- constraint
- estimate
- uncertainty
- confidence
- as_of_time

Después dibuja: $$\begin{array}{c}\text{Historical distribution } p \\ + \quad \boxed{\text{view} + \text{uncertainty}} \\ \downarrow \\ q. \end{array}$$

No implementes Entropy Pooling.

Responde en máximo cinco líneas:

¿por qué sería conceptualmente incorrecto que Research A y Research B deformaran \(p\) exactamente en la misma magnitud sólo porque expresan la misma dirección de view?

Finalmente propón qué debería ocurrir en los extremos:

$$ confidence\rightarrow0 \quad\Rightarrow\quad q\rightarrow ? $$ $$ confidence\rightarrow1 \quad\Rightarrow\quad q\rightarrow ? $$

No necesitas una fórmula; interpreta económicamente ambos límites.